### Chapter 7.4 - Multiple Input and Multiple Output Channels

Real CNN layers usually consume multiple input channels and produce multiple output channels.


In [ ]:
import torch
from torch import nn


### 7.4.1 Multiple Input Channels

#### 1. Intuition

With multiple input channels, each output value combines evidence from every channel in the local window.

The kernel has one small spatial grid per input channel.

#### 2. Why this exists

Color images and hidden feature maps contain multiple measurements at each location, so a useful detector often needs to combine channels.


#### 3. Examples

Inspect a kernel for three input channels.


In [ ]:
conv = nn.Conv2d(in_channels=3, out_channels=1, kernel_size=2)
conv.weight.shape


Apply it to one tiny color-like image.


In [ ]:
X = torch.zeros(1, 3, 4, 4)
Y = conv(X)
Y.shape


#### 4. Step-by-step breakdown

The weight shape is output channels, input channels, height, width.

There is one output channel here.

The kernel spans all three input channels.

The output has one feature map per image.

#### 5. Connection to ML systems

Each CNN layer learns how to mix feature channels from the previous layer.

#### 6. Common confusion points

- Input channels are consumed, not preserved automatically.
- A kernel for RGB images has depth 3.
- The spatial kernel size is separate from channel count.
- Channel order must match the framework convention.


### 7.4.2 Multiple Output Channels

#### 1. Intuition

Multiple output channels mean the layer learns multiple kernels.

Each output channel is a different learned feature map.

#### 2. Why this exists

One detector is not enough for vision. A layer may need edge detectors, color detectors, texture detectors, and many other patterns.


#### 3. Examples

Create four output channels.


In [ ]:
conv = nn.Conv2d(3, 4, kernel_size=3, padding=1)
X = torch.zeros(2, 3, 8, 8)
Y = conv(X)
Y.shape


Count weight values without bias.


In [ ]:
weights = 4 * 3 * 3 * 3
weights


#### 4. Step-by-step breakdown

The batch has 2 images.

The input has 3 channels.

The layer learns 4 output channels.

Padding preserves the 8 by 8 spatial size.

#### 5. Connection to ML systems

CNN width is often described by the number of channels in each layer.

#### 6. Common confusion points

- Output channels are chosen by the model designer.
- More channels usually mean more capacity and computation.
- Bias adds one value per output channel.
- Channel count is independent of batch size.


### 7.4.3 1 x 1 Convolutional Layer

#### 1. Intuition

A 1 x 1 convolution looks at one spatial location at a time but mixes the channel values at that location.

It does not combine neighboring pixels directly.

#### 2. Why this exists

1 x 1 convolutions are useful for changing channel count cheaply and mixing feature information across channels.


#### 3. Examples

Use a 1 x 1 convolution to change channel count.


In [ ]:
conv = nn.Conv2d(3, 2, kernel_size=1)
X = torch.zeros(1, 3, 5, 5)
Y = conv(X)
Y.shape


Count parameters without bias.


In [ ]:
params = 2 * 3 * 1 * 1
params


#### 4. Step-by-step breakdown

The layer reads 3 channel values at each location.

It writes 2 output channel values at the same location.

The 5 by 5 spatial size is unchanged.

Only channel mixing occurs.

#### 5. Connection to ML systems

1 x 1 convolutions are central to many modern CNN blocks, including NiN, GoogLeNet, ResNet bottlenecks, and MobileNet-style designs.

#### 6. Common confusion points

- 1 x 1 does not mean one parameter total.
- It has input-channel times output-channel weights.
- It mixes channels, not neighboring spatial positions.
- It is often used before or after larger spatial convolutions.


### 7.4.4 Discussion

#### 1. Intuition

Channels are the feature dimension of a CNN.

Spatial dimensions say where information is. Channel dimensions say what kind of information is stored there.

#### 2. Why this exists

Understanding channel flow is necessary before reading modern CNN architectures.


#### 3. Examples

Track channel counts through a small stack.


In [ ]:
net = nn.Sequential(
    nn.Conv2d(1, 4, 3, padding=1),
    nn.ReLU(),
    nn.Conv2d(4, 8, 3, padding=1),
)
net(torch.zeros(1, 1, 6, 6)).shape


#### 4. Step-by-step breakdown

The first convolution maps 1 channel to 4.

ReLU keeps the same shape.

The second convolution maps 4 channels to 8.

Padding keeps the spatial size at 6 by 6.

#### 5. Connection to ML systems

Architecture diagrams often list only channel counts and spatial sizes because these define the tensor flow.

#### 6. Common confusion points

- ReLU changes values but not shape.
- Convolutions decide output channels.
- Padding and stride decide spatial size.
- Shape tracking is a core CNN skill.


### 7.4.5 Exercises

#### 1. Intuition

These exercises practice channel counting.

#### 2. Why this exists

Most CNN shape mistakes come from confusing batch, channel, height, and width.


#### 3. Examples

Exercise 1: count weights in a 5-output-channel layer.


In [ ]:
out_ch, in_ch, k = 5, 3, 3
weights = out_ch * in_ch * k * k
weights


Exercise 2: inspect a 1 x 1 convolution weight shape.


In [ ]:
layer = nn.Conv2d(8, 4, kernel_size=1)
layer.weight.shape


#### 4. Step-by-step breakdown

Exercise 1 multiplies output channels, input channels, and kernel area.

Exercise 2 shows the same rule for a 1 x 1 kernel.

The spatial kernel area is one.

The channel mixing still has many weights.

#### 5. Connection to ML systems

These calculations prepare for modern CNN block designs.

#### 6. Common confusion points

- Always identify input channels before creating a Conv2d layer.
- Output channels become input channels for the next layer.
- Kernel size does not include batch size.
- Bias count equals output channels if bias is enabled.
